# Application of Artificial Intelligence in Personalized Learning Systems for Improving Student Performance

**Student:** Sajib Sarker · **Department:** Software Engineering · **Daffodil International University**

End-to-end notebook: OULAD data → feature engineering → leakage analysis → model comparison
(Logistic Regression, Random Forest, XGBoost) → SHAP & LIME explanations → export of website artifacts.

### How to run
1. Open in **Google Colab**.
2. Make sure the 7 OULAD CSV files are in `MyDrive/Thesis Dataset`.
3. **Runtime → Run all.** (≈10–15 min; the 10.6M-row `studentVle.csv` takes the longest.)
4. At the end, `website_artifacts.zip` is saved to Drive and downloaded automatically.

### Feature sets compared
| Name | Features | Note |
|---|---|---|
| `full40` | 40 | Original experiment (85.93% test). Contains leakage features — **reference only**. |
| `clean37` | 37 | Original clean experiment (73.02% test). Removes the 2 leakage features and `date_registration`. |
| `clean38` | 38 | Removes only the 2 leakage features. `date_registration` is known before the course starts, so it is kept. |

The website model is chosen **only from the leakage-free sets**, using **validation** accuracy (the test set is never used for selection).

## 0. Setup

In [ ]:
# Install explainability libraries (only needed in Colab; pandas/sklearn/xgboost are pre-installed)
import sys
if 'google.colab' in sys.modules:
    !pip install -q shap lime

In [ ]:
import os, json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn, xgboost

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

# ---- Mount Google Drive (Colab only) ----
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ---- Paths ----
DATA_DIR     = os.environ.get('OULAD_DIR', '/content/drive/MyDrive/Thesis Dataset')
OUTPUT_DIR   = os.path.join(DATA_DIR, 'thesis_outputs')       # figures + tables for the thesis
ARTIFACT_DIR = os.path.join(DATA_DIR, 'website_artifacts')    # files the website needs
FIG_DIR      = os.path.join(OUTPUT_DIR, 'figures')
for d in [OUTPUT_DIR, ARTIFACT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# ---- Global settings ----
RANDOM_STATE = 42
RUN_TUNING   = False   # True = re-tune XGBoost on the leakage-free sets (adds ~15 min)

def save_fig(name):
    # Save current matplotlib figure for the thesis report
    plt.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches='tight')

print('Data folder :', DATA_DIR)
print('Versions    : xgboost', xgboost.__version__, '| sklearn', sklearn.__version__,
      '| pandas', pd.__version__, '| numpy', np.__version__)

## 1. Load OULAD data

In [ ]:
student_info         = pd.read_csv(f'{DATA_DIR}/studentInfo.csv')
student_assessment   = pd.read_csv(f'{DATA_DIR}/studentAssessment.csv')
assessments          = pd.read_csv(f'{DATA_DIR}/assessments.csv')
student_vle          = pd.read_csv(f'{DATA_DIR}/studentVle.csv')
student_registration = pd.read_csv(f'{DATA_DIR}/studentRegistration.csv')
vle                  = pd.read_csv(f'{DATA_DIR}/vle.csv')
courses              = pd.read_csv(f'{DATA_DIR}/courses.csv')

for name, df in [('student_info', student_info), ('student_assessment', student_assessment),
                 ('assessments', assessments), ('student_vle', student_vle),
                 ('student_registration', student_registration), ('vle', vle), ('courses', courses)]:
    print(f'{name:22s}', df.shape)

In [ ]:
# Target distribution in the raw studentInfo table (all 32,593 registrations)
raw_dist = pd.DataFrame({
    'count': student_info['final_result'].value_counts(),
    'percent': student_info['final_result'].value_counts(normalize=True).mul(100).round(2)
})
print(raw_dist)

## 2. Assessment features

In [ ]:
# Score is stored as text in some versions -> convert to numeric ('?' becomes NaN)
student_assessment['score'] = pd.to_numeric(student_assessment['score'], errors='coerce')

# Attach assessment type (TMA / CMA / Exam), weight and deadline to every submission
assessment_data = student_assessment.merge(assessments, on='id_assessment', how='left')

KEYS = ['code_module', 'code_presentation', 'id_student']

# One row per student per module-presentation
assessment_features = assessment_data.groupby(KEYS).agg(
    avg_score=('score', 'mean'),
    max_score=('score', 'max'),
    min_score=('score', 'min'),
    score_std=('score', 'std'),
    assessment_count=('score', 'count'),
    avg_weight=('weight', 'mean'),
    total_weight=('weight', 'sum'),
    avg_submission_day=('date_submitted', 'mean')
).reset_index()

# std is NaN when a student has only one assessment -> 0 variation
assessment_features['score_std'] = assessment_features['score_std'].fillna(0)
print('Assessment features:', assessment_features.shape)

## 3. VLE (online engagement) features

In [ ]:
# Note: VLE clicks measure online learning engagement, NOT classroom attendance
vle_features = student_vle.groupby(KEYS).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    learning_sites=('id_site', 'nunique'),
    avg_clicks_per_record=('sum_click', 'mean'),
    max_clicks_per_record=('sum_click', 'max')
).reset_index()

print('VLE features:', vle_features.shape)
del student_vle   # free ~1 GB of RAM

## 4. Merge with student information

In [ ]:
# Outer join keeps students who have assessments OR VLE activity (or both).
# Students with neither are not included -> modelling set is 29,278 rows, not 32,593.
rich_data = assessment_features.merge(vle_features, on=KEYS, how='outer')
rich_data = rich_data.merge(student_info, on=KEYS, how='left')
print('Merged shape:', rich_data.shape)

## 5. Registration features

In [ ]:
reg = student_registration.copy()
reg['date_registration']   = pd.to_numeric(reg['date_registration'], errors='coerce')
reg['date_unregistration'] = pd.to_numeric(reg['date_unregistration'], errors='coerce')
reg['registration_duration'] = reg['date_unregistration'] - reg['date_registration']

rich_data = rich_data.merge(
    reg[KEYS + ['date_registration', 'date_unregistration', 'registration_duration']],
    on=KEYS, how='left'
)

# 1 if the student formally unregistered from the module
rich_data['has_unregistration'] = rich_data['date_unregistration'].notna().astype(int)
rich_data = rich_data.drop(columns=['date_unregistration'])

# Median imputation (kept identical to the original experiment for reproducibility)
rich_data['registration_duration'] = rich_data['registration_duration'].fillna(rich_data['registration_duration'].median())
rich_data['date_registration']     = rich_data['date_registration'].fillna(rich_data['date_registration'].median())

print('Shape after registration features:', rich_data.shape)

## 6. Handle missing assessment / VLE values

In [ ]:
# Missing aggregate = no recorded activity -> 0
assessment_cols = ['avg_score', 'max_score', 'min_score', 'score_std', 'assessment_count',
                   'avg_weight', 'total_weight', 'avg_submission_day']
vle_cols = ['total_clicks', 'active_days', 'learning_sites',
            'avg_clicks_per_record', 'max_clicks_per_record']

rich_data[assessment_cols] = rich_data[assessment_cols].fillna(0)
rich_data[vle_cols] = rich_data[vle_cols].fillna(0)

## 7. TMA and CMA features

In [ ]:
# TMA = Tutor Marked Assessment, CMA = Computer Marked Assessment
def type_features(a_type, prefix):
    sub = assessment_data[assessment_data['assessment_type'] == a_type]
    return sub.groupby(KEYS).agg(**{
        f'{prefix}_avg_score': ('score', 'mean'),
        f'{prefix}_max_score': ('score', 'max'),
        f'{prefix}_min_score': ('score', 'min'),
        f'{prefix}_count':     ('score', 'count'),
    }).reset_index()

rich_data = rich_data.merge(type_features('TMA', 'tma'), on=KEYS, how='left')
rich_data = rich_data.merge(type_features('CMA', 'cma'), on=KEYS, how='left')

tma_cma_cols = ['tma_avg_score', 'tma_max_score', 'tma_min_score', 'tma_count',
                'cma_avg_score', 'cma_max_score', 'cma_min_score', 'cma_count']
rich_data[tma_cma_cols] = rich_data[tma_cma_cols].fillna(0)

## 8. Derived features

In [ ]:
# Score consistency
rich_data['tma_score_range'] = rich_data['tma_max_score'] - rich_data['tma_min_score']
rich_data['cma_score_range'] = rich_data['cma_max_score'] - rich_data['cma_min_score']
rich_data['continuous_avg_score'] = (rich_data['tma_avg_score'] + rich_data['cma_avg_score']) / 2

# Engagement intensity (replace 0 with 1 to avoid division by zero)
rich_data['clicks_per_active_day'] = rich_data['total_clicks'] / rich_data['active_days'].replace(0, 1)
rich_data['sites_per_active_day']  = rich_data['learning_sites'] / rich_data['active_days'].replace(0, 1)
rich_data['click_consistency']     = rich_data['avg_clicks_per_record'] / rich_data['max_clicks_per_record'].replace(0, 1)

print('Final dataset shape:', rich_data.shape)
print('Remaining missing values:', int(rich_data.isnull().sum().sum()))

## 9. Final dataset and feature sets

In [ ]:
TARGET = 'final_result'
target = rich_data[TARGET]
features_all = rich_data.drop(columns=[TARGET, 'id_student'])   # 40 input features

CAT_COLS = ['code_module', 'code_presentation', 'gender', 'region',
            'highest_education', 'imd_band', 'age_band', 'disability']
assert rich_data[CAT_COLS].isnull().sum().sum() == 0, 'Unexpected missing categorical values'

LEAKY = ['has_unregistration', 'registration_duration']   # derived from date_unregistration

FEATURE_SETS = {
    'full40':  list(features_all.columns),
    'clean37': [c for c in features_all.columns if c not in LEAKY + ['date_registration']],
    'clean38': [c for c in features_all.columns if c not in LEAKY],
}
LEAKAGE_FREE = ['clean37', 'clean38']

for k, v in FEATURE_SETS.items():
    print(f'{k:8s}: {len(v)} features')

print('\nModelling-set class distribution (report THIS in the thesis):')
print(pd.DataFrame({'count': target.value_counts(),
                    'percent': target.value_counts(normalize=True).mul(100).round(2)}))

# Save processed dataset for reproducibility
rich_data.to_csv(os.path.join(OUTPUT_DIR, 'processed_dataset.csv'), index=False)

## 10. Leakage analysis
`date_unregistration` is recorded only for students who left the module, so `has_unregistration`
(and `registration_duration`, computed from it) almost directly reveals the **Withdrawn** label.

In [ ]:
leak_table = pd.crosstab(rich_data['has_unregistration'], rich_data[TARGET])
print(leak_table)
share = leak_table.loc[1, 'Withdrawn'] / leak_table.loc[1].sum() * 100 if 1 in leak_table.index else 0
print(f'\nStudents with has_unregistration = 1 who are Withdrawn: {share:.2f}%')
leak_table.to_csv(os.path.join(OUTPUT_DIR, 'leakage_crosstab.csv'))

## 11. Train / validation / test split (70 / 15 / 15)

In [ ]:
# Identical split to the original experiments (same random_state and stratification)
X_train_all, X_temp, y_train, y_temp = train_test_split(
    features_all, target, test_size=0.30, random_state=RANDOM_STATE, stratify=target)
X_val_all, X_test_all, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

print('Train:', X_train_all.shape, '| Validation:', X_val_all.shape, '| Test:', X_test_all.shape)

# Encode target: Distinction=0, Fail=1, Pass=2, Withdrawn=3
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc   = label_encoder.transform(y_val)
y_test_enc  = label_encoder.transform(y_test)
CLASSES = list(label_encoder.classes_)
print('Classes:', CLASSES)

In [ ]:
def build_preprocessor(cols):
    # One-hot for categorical, standard scaling for numeric (same as original experiments)
    cat = [c for c in CAT_COLS if c in cols]
    num = [c for c in cols if c not in CAT_COLS]
    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
        ('num', StandardScaler(), num),
    ])

def to_dense(X):
    return X.toarray() if hasattr(X, 'toarray') else np.asarray(X)

# Encode every feature set once
DATA = {}
for set_name, cols in FEATURE_SETS.items():
    prep = build_preprocessor(cols)
    DATA[set_name] = {
        'cols': cols,
        'prep': prep,
        'Xtr': to_dense(prep.fit_transform(X_train_all[cols])),
        'Xva': to_dense(prep.transform(X_val_all[cols])),
        'Xte': to_dense(prep.transform(X_test_all[cols])),
    }
    print(f'{set_name:8s} -> {DATA[set_name]["Xtr"].shape[1]} encoded features')

## 12. (Optional) XGBoost tuning on the leakage-free sets
The original hyperparameters were tuned on an earlier 26-feature set.
Set `RUN_TUNING = True` in Section 0 to re-tune on the leakage-free sets using the **validation** set.

In [ ]:
BASE_XGB_PARAMS = dict(max_depth=4, learning_rate=0.05, n_estimators=300,
                       subsample=0.8, colsample_bytree=0.8)
XGB_PARAMS = {s: dict(BASE_XGB_PARAMS) for s in FEATURE_SETS}

if RUN_TUNING:
    grid = ParameterGrid({'max_depth': [4, 6, 8], 'learning_rate': [0.05, 0.1],
                          'n_estimators': [300, 500], 'subsample': [0.8, 1.0],
                          'colsample_bytree': [0.8, 1.0]})
    for s in LEAKAGE_FREE:
        best_acc, best_p = 0, None
        for p in grid:
            m = XGBClassifier(**p, random_state=RANDOM_STATE, eval_metric='mlogloss', n_jobs=-1)
            m.fit(DATA[s]['Xtr'], y_train_enc)
            acc = accuracy_score(y_val_enc, m.predict(DATA[s]['Xva']))
            if acc > best_acc:
                best_acc, best_p = acc, p
        XGB_PARAMS[s] = best_p
        print(f'{s}: best validation accuracy {best_acc*100:.2f}% with {best_p}')
else:
    print('Tuning skipped. Using:', BASE_XGB_PARAMS)

## 13. Model comparison: Logistic Regression vs Random Forest vs XGBoost

In [ ]:
def make_models(set_name):
    return {
        'Logistic Regression': LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=15, min_samples_split=5,
                                                min_samples_leaf=2, class_weight='balanced',
                                                random_state=RANDOM_STATE, n_jobs=-1),
        'XGBoost': XGBClassifier(**XGB_PARAMS[set_name], random_state=RANDOM_STATE,
                                 eval_metric='mlogloss', n_jobs=-1),
    }

def evaluate(model, X, y_true):
    pred = model.predict(X)
    proba = model.predict_proba(X)
    return {
        'Accuracy (%)': round(accuracy_score(y_true, pred) * 100, 2),
        'Precision (macro)': round(precision_score(y_true, pred, average='macro'), 4),
        'Recall (macro)': round(recall_score(y_true, pred, average='macro'), 4),
        'F1 (macro)': round(f1_score(y_true, pred, average='macro'), 4),
        'F1 (weighted)': round(f1_score(y_true, pred, average='weighted'), 4),
        'ROC-AUC (OvR macro)': round(roc_auc_score(y_true, proba, multi_class='ovr', average='macro'), 4),
    }

rows, MODELS = [], {}
for s in FEATURE_SETS:
    for model_name, model in make_models(s).items():
        model.fit(DATA[s]['Xtr'], y_train_enc)
        MODELS[(s, model_name)] = model
        val_m = evaluate(model, DATA[s]['Xva'], y_val_enc)
        test_m = evaluate(model, DATA[s]['Xte'], y_test_enc)
        rows.append({'Feature set': s, 'Model': model_name,
                     'Val Accuracy (%)': val_m['Accuracy (%)'], **{f'Test {k}': v for k, v in test_m.items()}})
        print(f'{s:8s} | {model_name:20s} | val {val_m["Accuracy (%)"]:6.2f}% | test {test_m["Accuracy (%)"]:6.2f}%')

comparison = pd.DataFrame(rows)
comparison.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=False)
comparison

In [ ]:
# Reproducibility check against the previously reported results (base parameters only)
expected = {'full40': 85.93, 'clean37': 73.02}
for s, exp in expected.items():
    got = comparison.query('`Feature set` == @s and Model == "XGBoost"')['Test Accuracy (%)'].iloc[0]
    status = 'OK' if abs(got - exp) < 0.05 else 'DIFFERENT (check tuning flag / data files)'
    print(f'{s:8s} XGBoost test accuracy: {got:.2f}% (previous {exp}%) -> {status}')

In [ ]:
# Grouped bar chart of test accuracy
pivot = comparison.pivot(index='Model', columns='Feature set', values='Test Accuracy (%)')[list(FEATURE_SETS)]
ax = pivot.plot(kind='bar', figsize=(9, 5), rot=0, width=0.75)
ax.set_ylabel('Test accuracy (%)'); ax.set_title('Model comparison across feature sets')
ax.set_ylim(0, 100); ax.legend(title='Feature set')
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f', fontsize=8)
save_fig('model_comparison.png'); plt.show()

## 14. Select the website (deployment) model
Chosen from the **leakage-free** feature sets only, by **validation accuracy** of XGBoost
(XGBoost is used for the website because it supports fast, exact SHAP TreeExplainer explanations).

In [ ]:
xgb_val = comparison[(comparison['Model'] == 'XGBoost') & (comparison['Feature set'].isin(LEAKAGE_FREE))]
DEPLOY_SET = xgb_val.sort_values('Val Accuracy (%)', ascending=False)['Feature set'].iloc[0]

deploy_model = MODELS[(DEPLOY_SET, 'XGBoost')]
deploy_prep  = DATA[DEPLOY_SET]['prep']
deploy_cols  = DATA[DEPLOY_SET]['cols']
Xte_dep      = DATA[DEPLOY_SET]['Xte']

print('Deployment feature set:', DEPLOY_SET, f'({len(deploy_cols)} features)')
print(xgb_val[['Feature set', 'Val Accuracy (%)', 'Test Accuracy (%)', 'Test F1 (macro)']].to_string(index=False))

best_overall = comparison[comparison['Feature set'] == DEPLOY_SET].sort_values('Val Accuracy (%)', ascending=False).iloc[0]
print(f'\nBest model on {DEPLOY_SET} by validation accuracy: {best_overall["Model"]}')

## 15. Detailed evaluation of the deployment model (test set)

In [ ]:
test_pred = deploy_model.predict(Xte_dep)
print(classification_report(y_test_enc, test_pred, target_names=CLASSES, digits=4))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test_enc, test_pred), display_labels=CLASSES).plot(
    ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion matrix — XGBoost ({DEPLOY_SET})')
save_fig('confusion_matrix_deploy.png'); plt.show()

In [ ]:
# One-vs-rest ROC curves
proba = deploy_model.predict_proba(Xte_dep)
y_bin = label_binarize(y_test_enc, classes=range(len(CLASSES)))
plt.figure(figsize=(6, 5))
for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
    plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc_score(y_bin[:, i], proba[:, i]):.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title(f'ROC curves — XGBoost ({DEPLOY_SET})'); plt.legend()
save_fig('roc_curves_deploy.png'); plt.show()

## 16. SHAP explanations
SHAP values are computed on the encoded features, then one-hot columns are summed back to their
original feature (e.g. all `region_*` columns → `region`) so the explanation is readable.

In [ ]:
import shap

enc_names = list(deploy_prep.get_feature_names_out())

def raw_feature(enc_name):
    # Map an encoded column name back to its original feature name
    kind, rest = enc_name.split('__', 1)
    if kind == 'num':
        return rest
    return max([c for c in CAT_COLS if rest.startswith(c + '_')], key=len)

ENC_TO_RAW = {e: raw_feature(e) for e in enc_names}
RAW_ORDER = deploy_cols

def aggregate_to_raw(sv_2d):
    # sv_2d: (n_samples, n_encoded) -> DataFrame (n_samples, n_raw)
    df = pd.DataFrame(sv_2d, columns=enc_names)
    return df.T.groupby(ENC_TO_RAW).sum().T[RAW_ORDER]

explainer = shap.TreeExplainer(deploy_model)
rng = np.random.RandomState(RANDOM_STATE)
idx = rng.choice(len(Xte_dep), size=min(1000, len(Xte_dep)), replace=False)
X_shap = Xte_dep[idx]

sv = explainer.shap_values(X_shap)
sv = np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)   # -> (n, features, classes)
print('SHAP array shape:', sv.shape)

In [ ]:
# Global importance: mean |SHAP| per original feature, per class
global_imp = pd.DataFrame({cls: aggregate_to_raw(sv[:, :, i]).abs().mean() for i, cls in enumerate(CLASSES)})
global_imp['Total'] = global_imp.sum(axis=1)
top = global_imp.sort_values('Total', ascending=False).head(15)

top[CLASSES].iloc[::-1].plot(kind='barh', stacked=True, figsize=(9, 6))
plt.xlabel('Mean |SHAP value|'); plt.title('Global feature importance (SHAP) by class')
save_fig('shap_global_importance.png'); plt.show()

global_imp.sort_values('Total', ascending=False).to_csv(os.path.join(OUTPUT_DIR, 'shap_global_importance.csv'))

In [ ]:
# Beeswarm plot per class (encoded features, direction of effect)
pretty = [n.split('__', 1)[1] for n in enc_names]
for i, cls in enumerate(CLASSES):
    shap.summary_plot(sv[:, :, i], X_shap, feature_names=pretty, max_display=12, show=False)
    plt.title(f'SHAP summary — class: {cls}')
    save_fig(f'shap_beeswarm_{cls}.png'); plt.show()

In [ ]:
# Local explanation for ONE test student (first student predicted as 'Fail', else the first student)
fail_id = CLASSES.index('Fail')
preds_all = deploy_model.predict(Xte_dep)
EXAMPLE_POS = int(np.where(preds_all == fail_id)[0][0]) if (preds_all == fail_id).any() else 0

x_one = Xte_dep[EXAMPLE_POS:EXAMPLE_POS + 1]
pred_cls = int(deploy_model.predict(x_one)[0])
sv_one = explainer.shap_values(x_one)
sv_one = np.stack(sv_one, axis=-1) if isinstance(sv_one, list) else np.asarray(sv_one)
contrib = aggregate_to_raw(sv_one[:, :, pred_cls]).iloc[0]
top_c = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(10)[::-1]

student_row = X_test_all.iloc[EXAMPLE_POS]
print('Student id     :', rich_data.loc[student_row.name, 'id_student'])
print('Actual result  :', y_test.iloc[EXAMPLE_POS])
print('Predicted      :', CLASSES[pred_cls])
print('Probabilities  :', dict(zip(CLASSES, np.round(deploy_model.predict_proba(x_one)[0], 3))))

plt.figure(figsize=(8, 5))
plt.barh(top_c.index, top_c.values, color=['#d62728' if v < 0 else '#2ca02c' for v in top_c.values])
plt.axvline(0, color='k', lw=0.8)
plt.xlabel(f'SHAP value toward "{CLASSES[pred_cls]}"')
plt.title('Local SHAP explanation — one student')
save_fig('shap_local_example.png'); plt.show()

## 17. LIME explanation (same student)
LIME works on the **original (raw) features** so the rules are human-readable
(e.g. `avg_score <= 45.2`). Categorical values are converted to integer codes for LIME and
converted back before prediction.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

CAT_IN_SET = [c for c in deploy_cols if c in CAT_COLS]
CAT_IDX = [deploy_cols.index(c) for c in CAT_IN_SET]
CAT_VALUES = {c: sorted(X_train_all[c].astype(str).unique()) for c in CAT_IN_SET}

def raw_to_lime(df):
    # DataFrame of raw features -> numeric array (categoricals as integer codes)
    out = df[deploy_cols].copy()
    for c in CAT_IN_SET:
        out[c] = out[c].astype(str).map({v: i for i, v in enumerate(CAT_VALUES[c])}).fillna(0)
    return out.astype(float).values

def lime_predict(arr):
    # Numeric LIME array -> class probabilities from the deployed pipeline
    df = pd.DataFrame(arr, columns=deploy_cols)
    for c in CAT_IN_SET:
        codes = df[c].round().clip(0, len(CAT_VALUES[c]) - 1).astype(int)
        df[c] = [CAT_VALUES[c][k] for k in codes]
    for c in deploy_cols:
        if c not in CAT_IN_SET:
            df[c] = df[c].astype(float)
    return deploy_model.predict_proba(to_dense(deploy_prep.transform(df)))

lime_explainer = LimeTabularExplainer(
    training_data=raw_to_lime(X_train_all),
    feature_names=deploy_cols,
    class_names=CLASSES,
    categorical_features=CAT_IDX,
    categorical_names={i: CAT_VALUES[c] for i, c in zip(CAT_IDX, CAT_IN_SET)},
    mode='classification',
    random_state=RANDOM_STATE,
)

lime_exp = lime_explainer.explain_instance(
    raw_to_lime(X_test_all.iloc[[EXAMPLE_POS]])[0], lime_predict,
    num_features=10, labels=[pred_cls])

for rule, weight in lime_exp.as_list(label=pred_cls):
    print(f'{weight:+.4f}  {rule}')

fig = lime_exp.as_pyplot_figure(label=pred_cls)
plt.title(f'LIME explanation — predicted "{CLASSES[pred_cls]}"')
save_fig('lime_local_example.png'); plt.show()

## 18. Export artifacts for the website
Everything the Streamlit app needs, plus metadata (feature lists, form options, metrics, library versions).

In [ ]:
# Model files
joblib.dump(deploy_model,  os.path.join(ARTIFACT_DIR, 'model.pkl'))
deploy_model.save_model(os.path.join(ARTIFACT_DIR, 'model.json'))   # portable XGBoost format (used by the website)
joblib.dump(deploy_prep,   os.path.join(ARTIFACT_DIR, 'preprocessor.pkl'))
joblib.dump(label_encoder, os.path.join(ARTIFACT_DIR, 'label_encoder.pkl'))

# Background data for SHAP / LIME in the website (raw training rows)
X_train_all[deploy_cols].sample(min(2000, len(X_train_all)), random_state=RANDOM_STATE).to_csv(
    os.path.join(ARTIFACT_DIR, 'background.csv'), index=False)

# Real test-set students for the website's "sample student" demo (never used for training)
test_students = rich_data.loc[X_test_all.index, ['id_student'] + deploy_cols + [TARGET]]
test_students.to_csv(os.path.join(ARTIFACT_DIR, 'test_students.csv'), index=False)

# Comparison table + leakage table for the "Research" page
comparison.to_csv(os.path.join(ARTIFACT_DIR, 'model_comparison.csv'), index=False)
leak_table.to_csv(os.path.join(ARTIFACT_DIR, 'leakage_crosstab.csv'))
global_imp.sort_values('Total', ascending=False).to_csv(os.path.join(ARTIFACT_DIR, 'shap_global_importance.csv'))

num_cols = [c for c in deploy_cols if c not in CAT_COLS]
dep_row = comparison[(comparison['Feature set'] == DEPLOY_SET) & (comparison['Model'] == 'XGBoost')].iloc[0]
metadata = {
    'feature_set': DEPLOY_SET,
    'model_type': 'XGBoost',
    'features': deploy_cols,
    'categorical_features': CAT_IN_SET,
    'numerical_features': num_cols,
    'excluded_leakage_features': LEAKY,
    'classes': CLASSES,
    'n_encoded_features': int(deploy_model.n_features_in_),
    'encoded_to_raw': ENC_TO_RAW,
    'hyperparameters': XGB_PARAMS[DEPLOY_SET],
    'metrics': {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                for k, v in dep_row.items() if k not in ['Feature set', 'Model']},
    'category_options': CAT_VALUES,
    'numeric_ranges': {c: {'min': float(X_train_all[c].min()), 'max': float(X_train_all[c].max()),
                           'median': float(X_train_all[c].median())} for c in num_cols},
    'dataset': {'rows': int(len(rich_data)), 'train': int(len(X_train_all)),
                'validation': int(len(X_val_all)), 'test': int(len(X_test_all)),
                'class_distribution': {k: int(v) for k, v in target.value_counts().items()}},
    'versions': {'xgboost': xgboost.__version__, 'scikit-learn': sklearn.__version__,
                 'pandas': pd.__version__, 'numpy': np.__version__, 'shap': shap.__version__},
}
with open(os.path.join(ARTIFACT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

# Quick reload test: the saved files must give the same predictions
m2 = joblib.load(os.path.join(ARTIFACT_DIR, 'model.pkl'))
p2 = joblib.load(os.path.join(ARTIFACT_DIR, 'preprocessor.pkl'))
reload_pred = m2.predict(to_dense(p2.transform(X_test_all[deploy_cols])))
assert (reload_pred == deploy_model.predict(Xte_dep)).all(), 'Reload check failed!'
print('Reload check passed.\n')

print('Website artifacts:')
for f in sorted(os.listdir(ARTIFACT_DIR)):
    print(f'  {f:32s} {os.path.getsize(os.path.join(ARTIFACT_DIR, f))/1024:8.1f} KB')

In [ ]:
# Zip the artifacts (saved next to the folder in Drive) and download in Colab
zip_path = shutil.make_archive(os.path.join(DATA_DIR, 'website_artifacts'), 'zip', ARTIFACT_DIR)
print('Zip saved to:', zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## 19. Summary

In [ ]:
print('=' * 60)
print('THESIS EXPERIMENT SUMMARY')
print('=' * 60)
print(f'Modelling dataset : {len(rich_data):,} students | split 70/15/15 (stratified, seed {RANDOM_STATE})')
print(f'Leakage features  : {LEAKY}')
print(f'Deployed model    : XGBoost on {DEPLOY_SET} ({len(deploy_cols)} features)')
print(f'  Validation acc  : {dep_row["Val Accuracy (%)"]:.2f}%')
print(f'  Test accuracy   : {dep_row["Test Accuracy (%)"]:.2f}%')
print(f'  Test macro F1   : {dep_row["Test F1 (macro)"]:.4f}')
print(f'  Test ROC-AUC    : {dep_row["Test ROC-AUC (OvR macro)"]:.4f}')
print(f'\nThesis figures/tables : {OUTPUT_DIR}')
print(f'Website artifacts     : {ARTIFACT_DIR}')

### Notes for the thesis
- The model uses **full-course (retrospective)** information such as total clicks and all assessment
  scores. It is **not** an early-warning model.
- `full40` results (≈86%) are reported only to demonstrate the effect of label leakage.
- Median imputation for registration fields was computed on the full dataset before splitting
  (kept to match the original experiment); its effect is negligible but should be mentioned.
- `continuous_avg_score` treats a missing CMA score as 0 for modules without CMAs.